加性注意力
能处理查询和键是不同长度的矢量：q是m维度，k是n维度。设hidden维度为h后，会分别使用 h*q 和 h*k 的这两个矩阵来处理q和n，得到两个维度为h的向量。
score(q, k) = w_v^T tanh(W_q q + W_k k)

In [ ]:
import torch
import math
from torch import nn

In [ ]:
class Additiveattention_weights(nn.Module):
    # key_size, 
    def __init__(self, query_size, key_size, hidden_size) -> None:
        super().__init__()
        self.wq = nn.Linear(query_size, hidden_size)
        self.wk = nn.Linear(key_size, hidden_size)
        self.wv = nn.Linear(hidden_size, 1)

    # quries.shape(len, query_size)
    # keys.shape(len, len - 1, key_size)
    # values.shape(len, len - 1, value_size)
    def forward(self, queries, keys, values):
        q = self.wq(queries)
        k = self.wk(keys)
        print("q.shape",q.shape)
        print("k.shape",k.shape)
        t = torch.tanh(q.unsqueeze(1) + k)
        print("t.shape",t.shape)
        v = self.wv(t)
        print("v.shape",v.shape)
        self.attention_weights = nn.functional.softmax(v, dim=1)
        preds = (self.attention_weights * values).sum(dim=1)
        print("attention_weights.shape",self.attention_weights.shape)

        return preds

In [ ]:
query_size = 4
key_size = 3
hidden_size = 5
value_size = 3
net = Additiveattention_weights(query_size, key_size, hidden_size)

def getKeysAndValues(n_train, a_size, a_values):
    a_values = a_values.repeat(n_train,1).reshape(n_train, n_train, a_size)
    values = a_values[(1 - torch.eye(n_train)).type(torch.bool)].reshape(n_train, n_train - 1, value_size)
    return values

n_train = 50
queries = torch.randn(n_train, query_size)
keys = torch.randn(n_train, key_size)
keys = getKeysAndValues(n_train, key_size, keys)


values = torch.randn(n_train, value_size)
values = getKeysAndValues(n_train, value_size, values)

net(queries, keys, values)



q.shape torch.Size([50, 5])
k.shape torch.Size([50, 49, 5])
t.shape torch.Size([50, 49, 5])
v.shape torch.Size([50, 49, 1])
attention.shape torch.Size([50, 49, 1])


tensor([[ 0.0060,  0.0332, -0.1999],
        [-0.0189,  0.0801, -0.1591],
        [-0.0265,  0.0541, -0.1616],
        [-0.0666,  0.0787, -0.1872],
        [-0.0360,  0.0776, -0.1813],
        [-0.0537,  0.0673, -0.1665],
        [-0.0171,  0.0606, -0.1630],
        [-0.0554,  0.0628, -0.1620],
        [ 0.0119,  0.0457, -0.1606],
        [-0.0103,  0.0541, -0.2296],
        [-0.0456,  0.0665, -0.1985],
        [-0.0464,  0.0317, -0.1748],
        [-0.0050,  0.0638, -0.2125],
        [-0.0237,  0.0281, -0.1896],
        [-0.0631,  0.0666, -0.2036],
        [-0.0562,  0.0879, -0.1488],
        [-0.0075,  0.0550, -0.1587],
        [-0.0291,  0.0246, -0.1702],
        [-0.0134,  0.0620, -0.1895],
        [ 0.0005,  0.0503, -0.1922],
        [-0.0028,  0.0317, -0.1392],
        [-0.0221,  0.0612, -0.1613],
        [-0.0737,  0.0467, -0.2205],
        [-0.0195,  0.0727, -0.1782],
        [-0.0265,  0.0508, -0.1645],
        [-0.0124,  0.0374, -0.1667],
        [-0.0325,  0.0302, -0.1586],
 